In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [3]:
Epilepsy_Cohort_Demo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_paired_Final")

In [4]:
Epilepsy_Cohort_Demo.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)



In [5]:
from pyspark.sql.functions import min, max
# Filter records and aggregate to find earliest TBI_date and latest EPI_date
filtered_dates_df = Epilepsy_Cohort_Demo.select(
    min("TBI_date").alias("earliest_TBI_date"),
    max("EPI_date").alias("latest_EPI_date")
)

# Display the filtered and aggregated DataFrame
filtered_dates_df.show(truncate=False)

+-------------------------+-------------------------+
|earliest_TBI_date        |latest_EPI_date          |
+-------------------------+-------------------------+
|1992-04-06T19:08:00+00:00|2022-08-25T18:47:00+00:00|
+-------------------------+-------------------------+



In [6]:
Epilepsy_Control_Demo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_demo_paired_Final_ld")

In [7]:
Epilepsy_Control_Demo.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)



In [8]:
from pyspark.sql.functions import min, max
# Filter records and aggregate to find earliest TBI_date and latest EPI_date
filtered_dates_df = Epilepsy_Control_Demo.select(
    min("TBI_date").alias("earliest_TBI_date"),
    max("latest_diagdate").alias("latest_latest_diagdate")
)

# Display the filtered and aggregated DataFrame
filtered_dates_df.show(truncate=False)

+-------------------------+-------------------------+
|earliest_TBI_date        |latest_latest_diagdate   |
+-------------------------+-------------------------+
|1991-11-05T15:21:00+00:00|2028-03-20T05:00:00+00:00|
+-------------------------+-------------------------+



In [21]:
from pyspark.sql.functions import min, max, col
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Define a window specification to partition by personid and order by latest_diagdate
window_spec = Window.partitionBy().orderBy(col("latest_diagdate").desc())

# Add row number column to identify the row with the maximum latest_diagdate
max_date_df = Epilepsy_Control_Demo.withColumn("rn", F.row_number().over(window_spec))

# Filter records with row number 1 (i.e., the maximum latest_diagdate)
filtered_dates_df = max_date_df.filter(col("rn") == 1).select(
    "personid",
    "latest_diagdate"
)

# Display the filtered DataFrame
filtered_dates_df.show(truncate=False)

+------------------------------------+-------------------------+
|personid                            |latest_diagdate          |
+------------------------------------+-------------------------+
|00b23d12-532c-464e-955e-a0526c86bae8|2028-03-20T05:00:00+00:00|
+------------------------------------+-------------------------+



In [26]:
condition_tab = spark.table("condition")

In [27]:
condition_tab.printSchema()

root
 |-- conditionid: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- conditioncode: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- codingSystemId: string (nullable = true)
 |    |    |    |-- primaryDisplay: string (nullable = true)
 |-- effectivedate: string (nullable = true)
 |-- asserteddate: string (nullable = true)
 |-- type: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodi

In [29]:
Count_Result1 = spark.sql("SELECT * FROM condition WHERE personid = '00b23d12-532c-464e-955e-a0526c86bae8' AND TO_TIMESTAMP(effectivedate, 'yyyy-MM-dd''T''HH:mm:ssXXX') = TIMESTAMP('2028-03-20T05:00:00+00:00')")
Count_Result1.show(3, truncate=False)

+-----------+--------+-----------+-------------+-------------+------------+----+--------------+---------------------+------------------+-------------------+------+----------+-----------+------------------+------+------+
|conditionid|personid|encounterid|conditioncode|effectivedate|asserteddate|type|classification|managementdisciplines|confirmationstatus|responsibleprovider|status|statusdate|billingrank|presentonadmission|source|tenant|
+-----------+--------+-----------+-------------+-------------+------------+----+--------------+---------------------+------------------+-------------------+------+----------+-----------+------------------+------+------+
+-----------+--------+-----------+-------------+-------------+------------+----+--------------+---------------------+------------------+-------------------+------+----------+-----------+------------------+------+------+



In [4]:
print(Epilepsy_Cohort_Demo.count())
print(Epilepsy_Control_Demo.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

152400


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1008863


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
FEpilepsy_Cohort_Commo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Commo_FinalProcessed")

In [5]:
FEpilepsy_Cohort_Commo.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|01d66783-10de-4d42-ab12-e11a37a756b9|I10          |
|05eac6ee-f043-4063-99c9-de7027804e36|H40          |
|074f8fd3-457c-44d4-9d73-cb7681ee2b9b|F41          |
|0814196e-0ef2-4957-9be9-f6ccea38c2c9|L29          |
|0d21b75f-f7e1-4cf9-b8ce-9592f9ca165d|Z79          |
|106885cc-63ef-4ae4-b49e-a5bd4d57f185|E05          |
|11db264e-012e-4715-bb7a-049343393013|R29          |
|1459b8eb-e829-4d4e-9ddc-1e1463bfda76|Z01          |
|16be80f1-ea7b-414c-87c2-efc743ebf73d|E11          |
|1874d366-b1bc-4ccd-9445-3dbbfdb096e9|R13          |
|19faad5c-5909-4612-be0a-924fe1ee67bd|H46          |
|1d532d18-1469-4a02-aea8-b7840e39deaf|N85          |
|1e3ba4f6-5dba-4b92-9024-cea8ff24b6ce|S27          |
|20aa0613-4361-416c-ab4d-6e8dc05243a7|T85          |
|22dc2b77-dcd0-4896-b1b5-64e68bdf6f9e|K72          |
|25f90c95-026f-40d4-bf83-fa941184326a|R51     

<IPython.core.display.Javascript object>

In [5]:
# Excluding Records having Commo-id as Nulls
from pyspark.sql.functions import col

# Filter records where Latest_modified_comorbidityid is not null
F1_Epilepsy_Cohort_Commo = FEpilepsy_Cohort_Commo.filter(col("comorbidityid").isNotNull())

# Display the filtered records
F1_Epilepsy_Cohort_Commo.show(truncate=False)

# Count the number of records in the filtered DataFrame
print("Number of records after filtering F1:", F1_Epilepsy_Cohort_Commo.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|01d66783-10de-4d42-ab12-e11a37a756b9|I10          |
|05eac6ee-f043-4063-99c9-de7027804e36|H40          |
|074f8fd3-457c-44d4-9d73-cb7681ee2b9b|F41          |
|0814196e-0ef2-4957-9be9-f6ccea38c2c9|L29          |
|0d21b75f-f7e1-4cf9-b8ce-9592f9ca165d|Z79          |
|106885cc-63ef-4ae4-b49e-a5bd4d57f185|E05          |
|11db264e-012e-4715-bb7a-049343393013|R29          |
|1459b8eb-e829-4d4e-9ddc-1e1463bfda76|Z01          |
|16be80f1-ea7b-414c-87c2-efc743ebf73d|E11          |
|1874d366-b1bc-4ccd-9445-3dbbfdb096e9|R13          |
|19faad5c-5909-4612-be0a-924fe1ee67bd|H46          |
|1d532d18-1469-4a02-aea8-b7840e39deaf|N85          |
|1e3ba4f6-5dba-4b92-9024-cea8ff24b6ce|S27          |
|20aa0613-4361-416c-ab4d-6e8dc05243a7|T85          |
|22dc2b77-dcd0-4896-b1b5-64e68bdf6f9e|K72          |
|25f90c95-026f-40d4-bf83-fa941184326a|R51     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of records after filtering F1: 7767000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
print(FEpilepsy_Cohort_Commo.count())
print(FEpilepsy_Cohort_Commo.distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

7929539


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5585510


<IPython.core.display.Javascript object>

In [11]:
##################################Remove Duplicates and Write- cohort-commo############################################################
D_Epilepsy_Cohort_Commo = FEpilepsy_Cohort_Commo.distinct()
D_Epilepsy_Cohort_Commo.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Processed_Commo_Cohort_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
FEpilepsy_Control_Commo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Commo_FinalProcessed")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
print(FEpilepsy_Control_Commo.count())
print(FEpilepsy_Control_Commo.distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

16125111


<IPython.core.display.Javascript object>

In [12]:
##################################Remove Duplicates and write - Control ############################################################
D_Epilepsy_Control_Commo = FEpilepsy_Control_Commo.distinct()
D_Epilepsy_Control_Commo.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Processed_Commo_Control_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
#Reading-> Finalized Cohort,control - Commo
Processed_Commo_Cohort_toStack = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Processed_Commo_Cohort_toStack")
Processed_Commo_Control_toStack = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Processed_Commo_Control_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Reading-> Commo-Cohort
Extract_Cohort_commo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_comorbidity_Paired")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
#Reading-> Commo-Control
Extract_Control_commo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_comorbidity_Paired")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
FEpilepsy_Cohort_Commo.printSchema()
Extract_Cohort_commo.printSchema()
FEpilepsy_Control_Commo.printSchema()
Extract_Control_commo.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [10]:
print(FEpilepsy_Cohort_Commo.distinct().count())
print(Extract_Cohort_commo.distinct().count())
print(FEpilepsy_Control_Commo.distinct().count())
print(Extract_Control_commo.distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

20499629


<IPython.core.display.Javascript object>

In [6]:
#Reading-> Commo-Cohort
Epilepsy_Cohort_commo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Commo_Cohort_toStack")

In [7]:
from pyspark.sql.functions import col, upper, when

# Assuming your DataFrame is named Epilepsy_Cohort_commo
# Select and transform the comorbidityid column
Epilepsy_Cohort_commo = Epilepsy_Cohort_commo.withColumn(
    "comorbidityid",
    when(col("comorbidityid").rlike("^[a-z]"), upper(col("comorbidityid"))).otherwise(col("comorbidityid"))
)

# Ensure all comorbidityid values start with uppercase alphabets
Epilepsy_Cohort_commo = Epilepsy_Cohort_commo.withColumn("comorbidityid", upper(col("comorbidityid")))

# Show the transformed DataFrame
Epilepsy_Cohort_commo.show(truncate=False)

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|252f8034-cf8d-4801-aced-83c7b5e08b89|S09          |
|721a199d-b4c9-47cc-a0fd-1549a98d9952|Z79          |
|b0f55e70-f323-4c97-af58-eb074df08790|Z86          |
|cbd8a94d-449a-4ddd-8f35-3a2cae215181|I77          |
|d9ae5567-dfb7-4b6a-9882-0d61d7a9f32a|Z85          |
|f21b7953-da80-4cb1-914f-9b7fee53219a|D51          |
|0ad09f0b-8b76-4ae8-b9ad-818ddcc87a08|H25          |
|46c5bbcb-a1d4-4853-973d-0d3e04c666af|S70          |
|79fae565-8639-4e7f-81d6-99aae40b32cc|S05          |
|6835a3d9-60a8-4f71-82e5-10125f4ece4f|S09          |
|af3ce154-a81b-4b26-82e2-045279df7c90|M54          |
|ddc6e071-3b2c-4f4f-800a-5fe886707757|J44          |
|58ae9e5c-f1eb-451e-af29-f1c77a8868a1|K56          |
|57e32d49-12fa-47a4-afcb-2855f64aad6f|G62          |
|81f08e0f-b5c0-4f27-a440-44b67d68c8ec|R06          |
|cfca6803-327b-4ecc-9ce8-a5bdad338bf5|W01     

In [45]:
print(Epilepsy_Cohort_commo.select("comorbidityid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1911


<IPython.core.display.Javascript object>

In [8]:
Epilepsy_Cohort_commo = Epilepsy_Cohort_commo.distinct()

In [8]:
Epilepsy_Cohort_commo.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [9]:
print(Epilepsy_Cohort_commo.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5544963


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
print(Epilepsy_Cohort_commo.count())
print(Epilepsy_Cohort_commo.select("comorbidityid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1911


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
#Reading-> Commo-Control
Epilepsy_Control_Commo = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Commo_Control_toStack")

In [10]:
print(Epilepsy_Cohort_commo.count())
print(Epilepsy_Control_Commo.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5544973


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

15947680


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from pyspark.sql.functions import col, upper, when

# Assuming your DataFrame is named Epilepsy_Cohort_commo
# Select and transform the comorbidityid column
Epilepsy_Control_Commo = Epilepsy_Control_Commo.withColumn(
    "comorbidityid",
    when(col("comorbidityid").rlike("^[a-z]"), upper(col("comorbidityid"))).otherwise(col("comorbidityid"))
)

# Ensure all comorbidityid values start with uppercase alphabets
Epilepsy_Control_Commo = Epilepsy_Control_Commo.withColumn("comorbidityid", upper(col("comorbidityid")))

# Show the transformed DataFrame
Epilepsy_Control_Commo.show(truncate=False)

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|1dc3cf85-6bc6-4cbd-a526-f9977e15a5f5|F10          |
|75323ed7-a90b-4b12-9b13-729ca21c5022|S32          |
|0fe3ddb9-a05a-4992-b329-76e9b757a294|S06          |
|12b537d9-1a58-4ad3-8c8a-e6bddd1a5a89|S06          |
|234b4415-587f-4f39-b056-0005c6c75461|S42          |
|4c4cfc04-0597-4f77-b3da-6836fc1277f2|F19          |
|828ab821-7c7b-46ee-ab49-fb47e920a583|W22          |
|1e170f4b-45a5-4315-91e5-8077963ddbf6|M54          |
|9e126e84-8757-48e1-86eb-05ce9cbe5ebb|S22          |
|d35cc155-4c64-441f-9b4b-c004d83d2d95|Y92          |
|ca35bbd1-a427-4b45-b9e1-4ad3265da767|J15          |
|dbd3d46a-483c-4203-9e59-7a1a7a53a473|J45          |
|2ec54b8b-1325-43fe-9e4f-f30bb1a22536|I60          |
|99acd71d-e04f-4a54-a181-f6f6795da53f|Y92          |
|d7f40298-7800-4277-b589-b02a3790e318|F03          |
|18775487-a21e-43de-b387-7c26ec91f0c3|J45     

In [11]:
Epilepsy_Control_Commo = Epilepsy_Control_Commo.distinct()

In [50]:
print(Epilepsy_Control_Commo.count())
print(Epilepsy_Control_Commo.select("comorbidityid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1930


<IPython.core.display.Javascript object>

In [9]:
Epilepsy_Cohort_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_Epilepsy_Cohort_Med_Stack")

In [10]:
Epilepsy_Cohort_Med.printSchema()

root
 |-- personid: string (nullable = true)
 |-- drugname: string (nullable = true)



In [13]:
Epilepsy_Control_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_Epilepsy_Control_Med_Stack")

In [ ]:
print(Epilepsy_Cohort_Med.count())
print(Epilepsy_Control_Med.count())

In [11]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cohort_Lab_ToStack")

In [12]:
Epilepsy_Cohort_Lab.printSchema()

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [7]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Control_Lab_ToStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
Epilepsy_Cohort_Lab.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|05b46377-fece-48be-a1f5-bad5bb7842fd|770-8  |Normal                    |2018-05-11 |
|07ea477b-5291-44d3-8b3a-b137abd708d5|718-7  |Normal                    |2014-12-24 |
|09387be8-5f66-4def-b75e-08290e155a43|4544-3 |Low                       |2020-10-27 |
|09d35d50-fd6f-4a64-85b1-a372fd199277|2498-4 |Normal                    |2019-06-10 |
|0a160575-7be7-48a3-8413-70d84b22108c|751-8  |High                      |2021-03-02 |
|0a2a84b8-b4bc-4aba-998b-8381748f0481|41276-7|Normal                    |2017-02-11 |
|0acb2551-6ec8-4d71-9994-70fbbf48676b|5778-6 |Abnormal                  |2020-12-16 |
|0d320e8f-4dca-49b9-a3b1-1dc46fa211f1|14627-4|Normal                    |2018-11-16 |
|0df15f4d-123c-4287-bb90-fdbf9be29747|5905-5 |Normal  

<IPython.core.display.Javascript object>

In [26]:
Epilepsy_Control_Lab.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|000058c9-4684-4a8a-9913-6a6ba01f8208|53797-7|2021-10-08                |null       |
|000086b2-3048-43ce-a370-25dfde6b7fa7|3094-0 |Normal                    |2017-10-21 |
|000151d8-40ee-46d4-860a-a757a06ff3b7|5905-5 |Normal                    |2021-09-18 |
|00019068-80ee-4e89-884e-72a73e77eae7|788-0  |Normal                    |2017-12-20 |
|000290a4-6823-491b-9145-518a5812281f|5902-2 |Normal                    |2022-03-22 |
|0005761f-eaa6-43f0-84f4-f023f0d682dd|1975-2 |Normal                    |2021-07-14 |
|0005761f-eaa6-43f0-84f4-f023f0d682dd|4544-3 |Normal                    |2021-07-14 |
|000605e3-f575-4d22-93b7-e2343af7d910|33037-3|Normal                    |2020-06-24 |
|00060bca-a396-410d-99c6-44ed33e116f0|5902-2 |High    

<IPython.core.display.Javascript object>

In [ ]:
print(Epilepsy_Cohort_Lab.count())
print(Epilepsy_Control_Lab.count())

In [7]:
Epilepsy_Cohort_Demo.printSchema()
Epilepsy_Cohort_Commo.printSchema()
Epilepsy_Cohort_Med.printSchema()
Epilepsy_Cohort_Lab.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- drugname: string (nullable = true)
 |-- ClassName: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [8]:
Epilepsy_Cohort_Demo.show(truncate=False)

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+----------+------+-------+
|personid                            |birthdate |conditioncode|EPI_date                 |TBI_date                 |age_of_TBI_diagnosis|age_at_EPI_diagnosis|race      |gender|mh_date|
+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+----------+------+-------+
|0456963c-f0d7-4a3a-9ec1-37d0be268fda|2016-03-11|1            |2021-03-05T16:12:06+00:00|2016-04-19T09:23:00+00:00|0.10588971000000001 |4.985685670833333   |White     |Female|59     |
|483df9fc-512f-443c-b900-0cff70cd1a3c|2020-01-06|1            |2020-02-15T21:47:56+00:00|2020-02-15T21:15:00+00:00|0.10990703416666665 |0.10996851333333334 |White     |Female|0      |
|648ab86a-893b-4cab-87b1-1c95c5923911|2021-01-19|1            |2021-03-22T18:21:

In [10]:
Epilepsy_Control_Demo.show(truncate=False)

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+---------------+------+-------+
|personid                            |birthdate |conditioncode|TBI_date                 |latest_diagdate          |age_of_TBI_diagnosis|race           |gender|mh_date|
+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+---------------+------+-------+
|f7fda37c-1215-4b9b-bb7f-7f4f8499f742|2021-10-19|1            |2021-11-03T01:31:00+00:00|2021-11-03T01:31:00+00:00|0.040492458333333335|Other_race     |Female|0      |
|9bd0642f-42e6-4e17-bb23-ad84d4b2c83c|2019-07-01|1            |2019-07-18T21:21:00+00:00|2022-02-25T17:05:00+00:00|0.0480902775        |White          |Male  |31     |
|e7c15b5b-bd60-4e9c-bdce-7fe806f08d9d|2019-06-19|1            |2019-07-11T04:00:00+00:00|2022-02-01T17:00:00+00:00|0.06227598583333333 |White          |Male  |3

In [13]:
# Find unique personid in Epilepsy_Cohort_Commo not present in Epilepsy_Cohort_Demo
unique_personid = Epilepsy_Cohort_Demo.select("personid").subtract(Epilepsy_Cohort_Commo.select("personid"))

# Show the unique personid
unique_personid.show(truncate=False)
print("Count of Personid", unique_personid.count())

+--------+
|personid|
+--------+
+--------+

Count of Personid 0


In [14]:
from pyspark.sql.functions import when, lit
# Perform left join
Cohort_Demo_Como = Epilepsy_Cohort_Demo.join(Epilepsy_Cohort_commo.select("personid", "comorbidityid"), 
                                       "personid", "left")

# If personid is not present in Epilepsy_Cohort_Commo, mark comorbidityid as null
Cohort_Demo_Como = Cohort_Demo_Como.withColumn("comorbidityid", 
                                 when(Cohort_Demo_Como["comorbidityid"].isNull(), lit(None)).otherwise(Cohort_Demo_Como["comorbidityid"]))

# Show the final result
Cohort_Demo_Como.show(truncate=False)
print("Count of Cohort_Demo_Como", Cohort_Demo_Como.count())

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+-----+------+-------+-------------+
|personid                            |birthdate |conditioncode|EPI_date                 |TBI_date                 |age_of_TBI_diagnosis|age_at_EPI_diagnosis|race |gender|mh_date|comorbidityid|
+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+-----+------+-------+-------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|1964-03-10|1            |2015-05-13T12:11:00+00:00|2009-08-13T17:37:44+00:00|45.426705745        |51.17609580333333   |White|Female|69     |R20          |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|1964-03-10|1            |2015-05-13T12:11:00+00:00|2009-08-13T17:37:44+00:00|45.426705745        |51.17609580333333   |White|Female|69     |J40          |
|0053f0b8-4a78-4826-b193-9a92cd4e43

In [11]:
Cohort_Demo_Como.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [15]:
from pyspark.sql.functions import when, lit
# Perform left join
Control_Demo_Como = Epilepsy_Control_Demo.join(Epilepsy_Control_Commo.select("personid", "comorbidityid"), 
                                       "personid", "left")

# If personid is not present in Epilepsy_Cohort_Commo, mark comorbidityid as null
Control_Demo_Como = Control_Demo_Como.withColumn("comorbidityid", 
                                 when(Control_Demo_Como["comorbidityid"].isNull(), lit(None)).otherwise(Control_Demo_Como["comorbidityid"]))

# Show the final result
Control_Demo_Como.show(truncate=False)
print("Count of Control_Demo_Como", Control_Demo_Como.count())

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+-----+------+-------+-------------+
|personid                            |birthdate |conditioncode|TBI_date                 |latest_diagdate          |age_of_TBI_diagnosis|race |gender|mh_date|comorbidityid|
+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+-----+------+-------+-------------+
|0004c82c-aed4-4726-8efa-abb3edfbba69|1935-12-27|1            |2018-10-16T23:44:00+00:00|2018-10-26T15:21:00+00:00|82.80642174416667   |White|Male  |0      |S01          |
|0004c82c-aed4-4726-8efa-abb3edfbba69|1935-12-27|1            |2018-10-16T23:44:00+00:00|2018-10-26T15:21:00+00:00|82.80642174416667   |White|Male  |0      |I25          |
|0004c82c-aed4-4726-8efa-abb3edfbba69|1935-12-27|1            |2018-10-16T23:44:00+00:00|2018-10-26T15:21:00+00:00|82.80642174416667   |Whit

In [16]:
##################################Writing-StackedDemo-Como-Cohort############################################################
Cohort_Demo_Como.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Cohort_I1")

In [17]:
##################################Writing-StackedDemo-Como-Control############################################################
Control_Demo_Como.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Control_I1")

In [13]:
count_of_pid = Cohort_Demo_Como.select("personid").distinct().count()
print("Count of Pid", count_of_pid)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of Pid 152400


<IPython.core.display.Javascript object>

In [59]:
count_of_pid1 = Control_Demo_Como.select("personid").distinct().count()
print("Count of Pid1", count_of_pid1)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of Pid1 1008863


<IPython.core.display.Javascript object>

In [19]:
count_of_pid_original = Epilepsy_Cohort_Demo.select("personid").distinct().count()
print("Count of Pid from original dataframe", count_of_pid_original)

Count of Pid from original dataframe 152400


In [60]:
count_of_pid_original1 = Epilepsy_Control_Demo.select("personid").distinct().count()
print("Count of Pid from original dataframe", count_of_pid_original1)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of Pid from original dataframe 1008863


<IPython.core.display.Javascript object>

In [20]:
Cohort_Demo_Como.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [52]:
Commo_Val = Cohort_Demo_Como.select("comorbidityid").distinct()
print("Total No of Unique Commorbidities", Commo_Val.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total No of Unique Commorbidities 1912


<IPython.core.display.Javascript object>

In [61]:
Commo_Val1 = Control_Demo_Como.select("comorbidityid").distinct()
print("Total No of Unique Commorbidities", Commo_Val1.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total No of Unique Commorbidities 1931


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
# from pyspark.sql.functions import col, when

# # Extract unique non-null comorbidityid values
# unique_non_null_comorbidity_ids = Cohort_Demo_Como.select('comorbidityid').distinct().na.drop().rdd.flatMap(lambda x: x).collect()

# # Define a list of column expressions for marking 1 if comorbidityid exists for a personid, else mark 0
# column_exprs = [
#     when(col('comorbidityid') == comorbidityid, 1)
#     .otherwise(0)
#     .alias(comorbidityid) for comorbidityid in unique_non_null_comorbidity_ids
# ]

# # Apply column expressions to mark 1 if comorbidityid exists for a personid, else mark 0
# transformedDF = Cohort_Demo_Como.select(
#     col("personid"), 
#     *column_exprs
# )

# # Display transformed DataFrame
# transformedDF.show(truncate=False)

from pyspark.sql.functions import col, when

# Extract unique non-null comorbidityid values
unique_non_null_comorbidity_ids = Cohort_Demo_Como.select('comorbidityid').distinct().na.drop().rdd.flatMap(lambda x: x).collect()

# Define a list of column expressions for marking 1 if comorbidityid exists for a personid, else mark 0
column_exprs = [
    when(col('comorbidityid') == comorbidityid, 1)
    .otherwise(0)
    .alias(comorbidityid) for comorbidityid in unique_non_null_comorbidity_ids
]

# Apply column expressions to mark 1 if comorbidityid exists for a personid, else mark 0
transformedDF = Cohort_Demo_Como.select(
    *[col(c) for c in Cohort_Demo_Como.columns if c != 'comorbidityid'],  # Include all columns except comorbidityid
    *column_exprs
)

# Display transformed DataFrame
transformedDF.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+-----+------+-------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+--

<IPython.core.display.Javascript object>

In [54]:
transformedDF = transformedDF.drop("conditioncode")

▸,:,


In [49]:
count_of_pid_original = transformedDF.select("personid").distinct().count()
print("Count of Pid from original dataframe", count_of_pid_original)

Count of Pid from original dataframe 152400


In [34]:
transformedDF.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- cid_M19: integer (nullable = false)
 |-- cid_S68: integer (nullable = false)
 |-- cid_B05: integer (nullable = false)
 |-- cid_Z21: integer (nullable = false)
 |-- cid_Y30: integer (nullable = false)
 |-- cid_A23: integer (nullable = false)
 |-- cid_H82: integer (nullable = false)
 |-- cid_R16: integer (nullable = false)
 |-- cid_V89: integer (nullable = false)
 |-- cid_I31: integer (nullable = false)
 |-- cid_Q61: integer (nullable = false)
 |-- cid_V72: integer (nullable = false)
 |-- cid_O12: integer (nullable = false)
 |-- cid_X76: integer (nullable = false)
 |-- cid_Z12: integer (nullable = false)
 |-

In [35]:
transformedDF = transformedDF.distinct()

▸,:,


In [19]:
count_of_pid_original = transformedDF.select("personid").distinct().count()
print("Count of Pid from original dataframe", count_of_pid_original)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of Pid from original dataframe 152400


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Assuming df is your Spark DataFrame
num_columns1 = len(Epilepsy_Cohort_Demo.columns)
print("Total number of columns in Epilepsy_Cohort_Demo:", num_columns1)
# Assuming df is your Spark DataFrame
num_columns2 = len(Epilepsy_Cohort_commo.columns)
print("Total number of columns in Epilepsy_Cohort_commo:", num_columns2)

▸,:,


Total number of columns in Epilepsy_Cohort_Demo: 10
Total number of columns in Epilepsy_Cohort_commo: 2


In [55]:
num_columns = len(transformedDF.columns)
distinct_columns = len(set(transformedDF.columns))

# Print the results
print("Total number of columns:", num_columns)
print("Total number of distinct columns:", distinct_columns)

▸,:,


Total number of columns: 1920
Total number of distinct columns: 1920


In [26]:
# Assuming df is your DataFrame
num_columns = len(transformedDF.schema)
print("Total number of columns:", num_columns)

▸,:,


Total number of columns: 1935


In [37]:
transformedDF.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- cid_M19: integer (nullable = false)
 |-- cid_S68: integer (nullable = false)
 |-- cid_B05: integer (nullable = false)
 |-- cid_Z21: integer (nullable = false)
 |-- cid_Y30: integer (nullable = false)
 |-- cid_A23: integer (nullable = false)
 |-- cid_H82: integer (nullable = false)
 |-- cid_R16: integer (nullable = false)
 |-- cid_V89: integer (nullable = false)
 |-- cid_I31: integer (nullable = false)
 |-- cid_Q61: integer (nullable = false)
 |-- cid_V72: integer (nullable = false)
 |-- cid_O12: integer (nullable = false)
 |-- cid_X76: integer (nullable = false)
 |-- cid_Z12: integer (nullable = false)
 |-

In [39]:
transformedDF.show(1, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+----------+-------------------------+-------------------------+--------------------+--------------------+-----+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+----

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [62]:
from pyspark.sql.functions import col, when

# Extract unique non-null comorbidityid values
unique_non_null_comorbidity_ids1 = Control_Demo_Como.select('comorbidityid').distinct().na.drop().rdd.flatMap(lambda x: x).collect()

# Define a list of column expressions for marking 1 if comorbidityid exists for a personid, else mark 0
column_exprs = [
    when(col('comorbidityid') == comorbidityid, 1)
    .otherwise(0)
    .alias(comorbidityid) for comorbidityid in unique_non_null_comorbidity_ids1
]

# Apply column expressions to mark 1 if comorbidityid exists for a personid, else mark 0
transformedDFCTL = Control_Demo_Como.select(
    *[col(c) for c in Control_Demo_Como.columns if c != 'comorbidityid'],  # Include all columns except comorbidityid
    *column_exprs
)

# Display transformed DataFrame
transformedDFCTL.show(truncate=False)
transformedDFCTL = transformedDFCTL.drop("conditioncode")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+-----+------+-------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---

<IPython.core.display.Javascript object>

In [56]:
##################################Writing-StackedDemo-Como-Cohort############################################################
transformedDF.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Cohort_S1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
##################################Writing-StackedDemo-Como-Control############################################################
transformedDFCTL.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Control_S1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from pyspark.sql.functions import col, to_date

# Define the specific values for race, gender, mh_date, birthdate, personid, EPI_date, TBI_date, age_of_TBI_diagnosis, and age_at_EPI_diagnosis
race_value = "White"
gender_value = "Female"
mh_date_value = 69
birthdate_value = "1964-03-10"
personid_value = "0053f0b8-4a78-4826-b193-9a92cd4e43d5"
EPI_date_value = "2015-05-13"
TBI_date_value = "2009-08-13"
age_of_TBI_diagnosis_value = 45.426705745
age_at_EPI_diagnosis_value = 51.17609580333333

# Convert string dates to date format
EPI_date_value = to_date(EPI_date_value)
TBI_date_value = to_date(TBI_date_value)

# Filter records with specific values for race, gender, mh_date, birthdate, personid, EPI_date, TBI_date, age_of_TBI_diagnosis, and age_at_EPI_diagnosis
filteredDF = transformedDF.filter(
    (col("race") == race_value) &
    (col("gender") == gender_value) &
    (col("mh_date") == mh_date_value) &
    (col("birthdate") == birthdate_value) &
    (col("personid") == personid_value) &
    (col("EPI_date") == EPI_date_value) &
    (col("TBI_date") == TBI_date_value) &
    (col("age_of_TBI_diagnosis") == age_of_TBI_diagnosis_value) &
    (col("age_at_EPI_diagnosis") == age_at_EPI_diagnosis_value)
)

# Display filtered DataFrame
filteredDF.show(truncate=False)

▸,:,


AnalysisException: "cannot resolve '`2015-05-13`' given input columns: [B49, B86, J67, K14, B54, C40, F43, Q28, A03, I78, NKP, H27, D41, H67, M92, E82, H44, K75, N03, S00, Q82, T59, D01, Z81, mh_date, W25, V29, Z47, J20, N15, Q45, Q23, X75, F82, T70, Y82, J81, W30, W46, B83, N91, Q62, A23, C02, O82, A53, S27, H49, R31, T57, R19, K90, A32, C34, A83, K27, K60, V19, N01, E54, O33, V58, T07, p79, Q04, S15, N71, B59, O15, I46, Y80, L92, I86, G32, Y22, O60, S77, M40, L55, L42, L51, Z79, N32, N73, I80, W16, G26, V35, N62, L72, L01, I31, Q55, C13, M00, G36, D52, R50, S82, D43, X73, N75, K56, N23, Q02, K22, H52, Q63, R17, N31, J98, Y27, S30, K31, K01, S63, C18, J40, M16, W55, J04, O09, M91, H82, Y07, F16, F07, C78, S62, N14, S11, N19, M80, D3A, N16, I77, H46, R51, I47, R93, F11, D13, R91, n43, J32, B05, Y00, B35, A98, O29, B41, K85, B69, Q92, T69, P19, Z91, V64, L29, F59, Q27, B97, G57, G52, P76, W31, R74, I96, T56, A36, X95, I67, K94, L57, C85, R36, I09, F90, T47, A45, Y77, O64, V93, race, K81

In [55]:
# Get the list of column names
column_names = transformedDF.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 1935


In [18]:
print(filteredDF.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

25


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from pyspark.sql.functions import col

# Stack the DataFrames
stacked_df = Epilepsy_Cohort_Lab.union(Epilepsy_Control_Lab)

# Remove duplicates
unique_lab_codes_df = stacked_df.select('labcode').distinct()

# Count the number of distinct lab codes
distinct_lab_code_count = unique_lab_codes_df.count()

# Display the count of distinct lab codes
print("Number of distinct lab codes:", distinct_lab_code_count)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of distinct lab codes: 10825


<IPython.core.display.Javascript object>

In [23]:
from pyspark.sql.functions import col

# Stack the DataFrames
stacked_df = Epilepsy_Cohort_Lab.union(Epilepsy_Control_Lab)

# Remove duplicates
unique_lab_interpret_df = stacked_df.select('New_updated_Interpretation').distinct()

# Count the number of distinct lab codes
distinct_lab_interpret_df = unique_lab_interpret_df.count()

# Display the count of distinct lab codes
print("Number of distinct lab interpret:", distinct_lab_interpret_df)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of distinct lab interpret: 6673


<IPython.core.display.Javascript object>

In [25]:
unique_lab_interpret_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------------+
|New_updated_Interpretation|
+--------------------------+
|2019-08-22                |
|2019-08-23                |
|2017-12-05                |
|2020-02-26                |
|2020-04-13                |
|2019-08-08                |
|2021-11-03                |
|2017-05-14                |
|2014-02-22                |
|2011-01-29                |
|2014-02-16                |
|2008-12-03                |
|2016-08-17                |
|2015-05-01                |
|2009-12-04                |
|2008-11-19                |
|2014-05-27                |
|2014-12-13                |
|2009-01-04                |
|2008-04-20                |
+--------------------------+
only showing top 20 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
# Remove duplicates
unique_lab_interpret_df = Epilepsy_Cohort_Lab.select('New_updated_Interpretation').distinct()

# Count the number of distinct lab codes
distinct_lab_interpret_df = unique_lab_interpret_df.count()

# Display the count of distinct lab codes
print("Number of distinct lab interpret:", distinct_lab_interpret_df)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of distinct lab interpret: 5982


<IPython.core.display.Javascript object>

In [2]:
Epilepsy_Cohort_Demo_Como = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Cohort_S1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
Epilepsy_Control_Demo_Como = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Control_S1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
Epilepsy_Cohort_Demo_Como.printSchema()
Epilepsy_Control_Demo_Como.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- M19: integer (nullable = true)
 |-- S68: integer (nullable = true)
 |-- Z21: integer (nullable = true)
 |-- Y30: integer (nullable = true)
 |-- A23: integer (nullable = true)
 |-- B05: integer (nullable = true)
 |-- H82: integer (nullable = true)
 |-- V89: integer (nullable = true)
 |-- R16: integer (nullable = true)
 |-- I31: integer (nullable = true)
 |-- Q61: integer (nullable = true)
 |-- V72: integer (nullable = true)
 |-- O12: integer (nullable = true)
 |-- X76: integer (nullable = true)
 |-- S39: integer (nullable = true)
 |-- Z12: integer (nullable = true)
 |-- L65: integer (nullable = true)
 |-- X04: integer (nullable = true)
 |-- F25: integ

In [30]:
Epilepsy_Cohort_Med.printSchema()
Epilepsy_Control_Med.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- drugname: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- drugname: string (nullable = true)



In [14]:
print(Epilepsy_Cohort_Demo_Como.count())

15947758


In [13]:
# from pyspark.sql.functions import col, when

# # Perform left join
# Epilepsy_Cohort_Demo_Como_Med = Epilepsy_Cohort_Demo_Como.join(Epilepsy_Cohort_Med, 
#                                             Epilepsy_Cohort_Demo_Como.personid == Epilepsy_Cohort_Med.personid, 
#                                             "left")

# # Select the desired columns and mark drugname as null where personid doesn't have a match
# Epilepsy_Cohort_Demo_Como_Med = Epilepsy_Cohort_Demo_Como_Med.select(Epilepsy_Cohort_Demo_Como["*"],
#                              when(col("Epilepsy_Cohort_Med.personid").isNull(), None).otherwise(col("Epilepsy_Cohort_Med.drugname")).alias("drugname"))

# Epilepsy_Cohort_Demo_Como_Med.show(truncate=False)
# print(Epilepsy_Cohort_Demo_Como_Med.count())
from pyspark.sql.functions import col, when
# Perform left join
Epilepsy_Cohort_Demo_Como_Med = Epilepsy_Cohort_Demo_Como.join(Epilepsy_Cohort_Med, 
                                            Epilepsy_Cohort_Demo_Como["personid"] == Epilepsy_Cohort_Med["personid"], 
                                            "left")

# Select the desired columns and mark drugname as null where personid doesn't have a match
Epilepsy_Cohort_Demo_Como_Med = Epilepsy_Cohort_Demo_Como_Med.select(Epilepsy_Cohort_Demo_Como["*"],
                             when(Epilepsy_Cohort_Med["personid"].isNull(), None).otherwise(Epilepsy_Cohort_Med["drugname"]).alias("drugname"))

Epilepsy_Cohort_Demo_Como_Med.show(truncate=False)
print(Epilepsy_Cohort_Demo_Como_Med.count())

+------------------------------------+----------+-------------------------+-------------------------+--------------------+-----+------+-------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-

15947758


In [ ]:
from pyspark.sql.functions import col, when

# Extract unique non-null comorbidityid values
unique_non_null_drugnames = Epilepsy_Cohort_Demo_Como_Med.select('drugname').distinct().na.drop().rdd.flatMap(lambda x: x).collect()

# Define a list of column expressions for marking 1 if comorbidityid exists for a personid, else mark 0
column_exprs = [
    when(col('drugname') == drugname, 1)
    .otherwise(0)
    .alias(drugname) for drugname in unique_non_null_drugnames
]

# Apply column expressions to mark 1 if comorbidityid exists for a personid, else mark 0
transformedDrugDF = Epilepsy_Cohort_Demo_Como_Med.select(
    *[col(c) for c in Epilepsy_Cohort_Demo_Como_Med.columns if c != 'drugname'],  # Include all columns except comorbidityid
    *column_exprs
)

# Display transformed DataFrame
transformedDrugDF.show(truncate=False)
print(transformedDrugDF.count())

In [21]:
Epilepsy_Cohort_Commo_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Med_Cohort_S3")

In [22]:
Epilepsy_Control_Commo_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Med_Control_S3")

In [3]:
Epilepsy_Cohort_Commo_Med.printSchema()
Epilepsy_Control_Commo_Med.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = true)
 |-- G12: long (nullable

In [4]:
# Assuming df is your DataFrame
num_columns = len(Epilepsy_Cohort_Commo_Med.schema)
print("Total number of columns:", num_columns)
# Assuming df is your DataFrame
num_columns1 = len(Epilepsy_Control_Commo_Med.schema)
print("Total number of columns1:", num_columns1)

Total number of columns: 2338
Total number of columns1: 2383


In [55]:
import numpy as np
from pyspark.sql.functions import col
from tabulate import tabulate

def cohort_distribution_Final_float(spark, cohort_demo):
    cohort_demo = cohort_demo.distinct()
    cohort_demo.cache()

    # Calculate gender counts and percentages
    genderCnt = cohort_demo.select('personid', 'gender').na.drop().distinct().groupBy('gender').count()
    total_gender_count = genderCnt.agg({'count': 'sum'}).collect()[0][0]
    genderCnt = genderCnt.withColumn('percentage', (col('count') / total_gender_count) * 100)

    # Calculate race counts and percentages
    raceCnt = cohort_demo.select('personid', 'race').na.drop().distinct().groupBy('race').count()
    total_race_count = raceCnt.agg({'count': 'sum'}).collect()[0][0]
    raceCnt = raceCnt.withColumn('percentage', (col('count') / total_race_count) * 100)

    # Calculate the total count of distinct individuals before age filtering
    total_individuals_count = cohort_demo.count()
    print('total_individuals_count', total_individuals_count)

    age_ranges = ['age<18', '18<=age<44', '44<=age<60', 'age>=60']  # Age ranges as strings
    age_statistics = []  # To store mean and standard deviation for each age range

    # Calculate counts, percentages, mean, and standard deviation for each age range
    for age_range in age_ranges:
        if age_range == 'age<18':
            age_data = cohort_demo.filter(cohort_demo.age_of_TBI_diagnosis < 18)
        elif age_range == '18<=age<44':
            age_data = cohort_demo.filter((cohort_demo.age_of_TBI_diagnosis >= 18) & (cohort_demo.age_of_TBI_diagnosis < 44))
        elif age_range == '44<=age<60':
            age_data = cohort_demo.filter((cohort_demo.age_of_TBI_diagnosis >= 44) & (cohort_demo.age_of_TBI_diagnosis < 60))
        elif age_range == 'age>=60':
            age_data = cohort_demo.filter(cohort_demo.age_of_TBI_diagnosis >= 60)

        age_count = age_data.count()

        # Calculate percentage using the total count before age filtering
        age_percentage = (age_count / total_individuals_count) * 100

        # Calculate sample statistics
        age_data = age_data.select('age_of_TBI_diagnosis').rdd.flatMap(lambda x: x).collect()
        sample_mean_age = np.mean(age_data)
        sample_std_dev_age = np.std(age_data, ddof=1)

        age_statistics.append([age_range, age_count, f"{age_percentage:.2f}%", f"{sample_mean_age:.2f}", f"{sample_std_dev_age:.2f}"])

    # Calculate mean and standard deviation across all age ranges
    all_age_data = cohort_demo.select('age_of_TBI_diagnosis').rdd.flatMap(lambda x: x).collect()
    all_age_mean = np.mean(all_age_data)
    all_age_std_dev = np.std(all_age_data, ddof=1)

    age_statistics.append(['All Ages', total_individuals_count, '100.00%', f"{all_age_mean:.2f}", f"{all_age_std_dev:.2f}"])

    # Print gender table
    print("Gender Table:")
    print(tabulate(genderCnt.toPandas(), headers=['Gender', 'Count', 'Percentage']))

    # Print race table
    print("Race Table:")
    print(tabulate(raceCnt.toPandas(), headers=['Race', 'Count', 'Percentage']))

    # Print age table
    print("Age Table:")
    print(tabulate(age_statistics, headers=['Age Range', 'Count', 'Percentage', 'Mean Age', 'Standard Deviation'], tablefmt="grid"))

# Example usage
# cohort_demo = spark.read.csv("path/to/your/dataset.csv", header=True, inferSchema=True)
# cohort_distribution_Final_float(cohort_demo)

▸,:,


In [6]:
cohort_distribution_Final_float(spark, Epilepsy_Cohort_Commo_Med)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Py4JJavaError: An error occurred while calling o147.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 3.0 failed 1 times, most recent failure: Lost task 5.0 in stage 3.0 (TID 8, localhost, executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.util.Arrays.copyOf(Arrays.java:3332)
	at java.lang.AbstractStringBuilder.ensureCapacityInternal(AbstractStringBuilder.java:124)
	at java.lang.AbstractStringBuilder.append(AbstractStringBuilder.java:448)
	at java.lang.StringBuilder.append(StringBuilder.java:136)
	at scala.StringContext.standardInterpolator(StringContext.scala:126)
	at scala.StringContext.s(StringContext.scala:95)
	at org.apache.spark.sql.catalyst.expressions.codegen.GenerateOrdering$.create(GenerateOrdering.scala:165)
	at org.apache.spark.sql.catalyst.expressions.codegen.GenerateOrdering$.create(GenerateOrdering.scala:55)
	at org.apache.spark.sql.catalyst.expressions.codegen.GenerateOrdering.create(GenerateOrderin

<IPython.core.display.Javascript object>

In [4]:
cohort_distribution_Final_float(spark, Epilepsy_Cohort_Demo)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    Race               Count    Percentage
--  ---------------  -------  ------------
 0  Native American     2047       1.34318
 1  Other_race         31664      20.7769
 2  White             109105      71.5912
 3  Hispanic            5626       3.6916
 4  Black               2009       1.31824
 5  Asian               1949       1.27887
Age Table:
+-------------+---------+--------------+------------+----------------------+
| Age Range   |   Count | Percentage   |   Mean Age |   Standard Deviation |
+=============+=========+==============+============+======================+
| age<18      |   40072 | 26.29%       |       7.98 |                 5.91 |
+-------------+---------+--------------+------------+----------------------+
| 18<=age<44  |   41417 | 27.18%       |      30.45 |                 7.49 |
+-------------+---------+--------------+------------+----------------------+
| 44<=age<60  |   26362 | 17.30%       |      52.36 |                 4.54 |
+-------------+---------+-------

<IPython.core.display.Javascript object>

In [56]:
cohort_distribution_Final_float(spark, Epilepsy_Control_Demo)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

    Race               Count    Percentage
--  ---------------  -------  ------------
 0  Native American     8018      0.794756
 1  Other_race        262024     25.9722
 2  White             675746     66.9809
 3  Hispanic           38984      3.86415
 4  Black               8702      0.862555
 5  Asian              15389      1.52538
Age Table:
+-------------+---------+--------------+------------+----------------------+
| Age Range   |   Count | Percentage   |   Mean Age |   Standard Deviation |
+=============+=========+==============+============+======================+
| age<18      |  494741 | 49.04%       |       8.22 |                 5.78 |
+-------------+---------+--------------+------------+----------------------+
| 18<=age<44  |  271093 | 26.87%       |      28.25 |                 7.43 |
+-------------+---------+--------------+------------+----------------------+
| 44<=age<60  |   91828 | 9.10%        |      51.99 |                 4.61 |
+-------------+---------+----------

<IPython.core.display.Javascript object>

In [3]:
print(Epilepsy_Cohort_Demo_Como.rdd.getNumPartitions())

▸,:,


10


In [4]:
# reparNum = Epilepsy_Cohort_Demo_Como.rdd.getNumPartitions()
Epilepsy_Cohort_Demo_Como_repar = Epilepsy_Cohort_Demo_Como.repartition(10)

▸,:,


In [7]:
cohort_distribution_Final_float(spark, Epilepsy_Cohort_Demo_Como_repar)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Py4JJavaError: An error occurred while calling o137.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 2.0 failed 1 times, most recent failure: Lost task 5.0 in stage 2.0 (TID 16, localhost, executor driver): java.lang.OutOfMemoryError: Java heap space

Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.org$apache$spark$scheduler$DAGScheduler$$failJobAndIndependentStages(DAGScheduler.scala:1889)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$abortStage$1.apply(DAGScheduler.scala:1877)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$abortStage$1.apply(DAGScheduler.scala:1876)
	at scala.collection.mutable.ResizableArray$class.foreach(ResizableArray.scala:59)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:48)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:1876)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$handleTaskSetFailed$1.apply(DAGScheduler.scala:926)
	at o

In [12]:
print(Epilepsy_Cohort_Demo.count())
print(Cohort_Demo_Como.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

152400


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5544976


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
Cohort_Demo_Como.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- conditioncode: string (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- mh_date: integer (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [46]:
############################Finalized :DS - cohort-Demo-Como##################################################################################
from pyspark.sql import functions as F

# Count the total number of unique comorbidity IDs
total_unique_comorbidity_ids = Cohort_Demo_Como.select('comorbidityid').distinct().count()

# Group by 'personid' and count the number of comorbidity IDs for each person
comorbidity_counts_per_person = (
    Cohort_Demo_Como
    .groupby('personid')
    .agg(F.countDistinct('comorbidityid').alias('num_comorbidities'))
)

# Calculate the minimum, maximum, mean, and standard deviation of the number of comorbidities per person
statistics = (
    comorbidity_counts_per_person
    .select(
        F.min('num_comorbidities').alias('min_comorbidities'),
        F.max('num_comorbidities').alias('max_comorbidities'),
        F.mean('num_comorbidities').alias('mean_comorbidities'),
        F.stddev('num_comorbidities').alias('stddev_comorbidities')
    )
    .collect()[0]
)

# Display the results
print("Total number of unique comorbidity IDs:", total_unique_comorbidity_ids)
print("Minimum number of comorbidities per person:", statistics['min_comorbidities'])
print("Maximum number of comorbidities per person:", statistics['max_comorbidities'])
print("Mean number of comorbidities per person:", statistics['mean_comorbidities'])
print("Standard deviation of comorbidities per person:", statistics['stddev_comorbidities'])

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total number of unique comorbidity IDs: 1912
Minimum number of comorbidities per person: 0
Maximum number of comorbidities per person: 377
Mean number of comorbidities per person: 36.38427165354331
Standard deviation of comorbidities per person: 27.573908225468582


<IPython.core.display.Javascript object>

In [47]:
############################Finalized :DS - control-Demo-Como##################################################################################
from pyspark.sql import functions as F

# Count the total number of unique comorbidity IDs
total_unique_comorbidity_ids = Control_Demo_Como.select('comorbidityid').distinct().count()

# Group by 'personid' and count the number of comorbidity IDs for each person
comorbidity_counts_per_person = (
    Control_Demo_Como
    .groupby('personid')
    .agg(F.countDistinct('comorbidityid').alias('num_comorbidities'))
)

# Calculate the minimum, maximum, mean, and standard deviation of the number of comorbidities per person
statistics = (
    comorbidity_counts_per_person
    .select(
        F.min('num_comorbidities').alias('min_comorbidities'),
        F.max('num_comorbidities').alias('max_comorbidities'),
        F.mean('num_comorbidities').alias('mean_comorbidities'),
        F.stddev('num_comorbidities').alias('stddev_comorbidities')
    )
    .collect()[0]
)

# Display the results
print("Total number of unique comorbidity IDs:", total_unique_comorbidity_ids)
print("Minimum number of comorbidities per person:", statistics['min_comorbidities'])
print("Maximum number of comorbidities per person:", statistics['max_comorbidities'])
print("Mean number of comorbidities per person:", statistics['mean_comorbidities'])
print("Standard deviation of comorbidities per person:", statistics['stddev_comorbidities'])

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total number of unique comorbidity IDs: 1931
Minimum number of comorbidities per person: 0
Maximum number of comorbidities per person: 211
Mean number of comorbidities per person: 15.807545722263578
Standard deviation of comorbidities per person: 14.732430439659948


<IPython.core.display.Javascript object>

In [45]:
from pyspark.sql import functions as F

# Group by comorbidity ID and count the number of distinct person IDs for each comorbidity ID
comorbidity_counts = (
    Cohort_Demo_Como
    .groupby('comorbidityid')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 1 comorbidity ID
top_1_comorbidity_id = (
    comorbidity_counts
    .orderBy(F.desc('count'))
    .limit(1)
)

# Show the top 1 comorbidity ID along with its count
top_1_comorbidity_id.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+-----+
|comorbidityid|count|
+-------------+-----+
|S09          |95417|
+-------------+-----+



<IPython.core.display.Javascript object>

In [44]:
##############################Finalized DS : Top 10 CommorbidityId-Cohort ###############################################
from pyspark.sql import functions as F

# Group by comorbidity ID and count the number of distinct person IDs for each comorbidity ID
comorbidity_counts = (
    Cohort_Demo_Como
    .groupby('comorbidityid')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 comorbidity IDs
top_10_comorbidity_ids = (
    comorbidity_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Calculate the total count of distinct person IDs present in the entire dataframe
total_count = (
    Cohort_Demo_Como
    .select(F.countDistinct('personid'))
    .collect()[0][0]
)

# Calculate the total count of distinct person IDs associated with the top 10 comorbidity IDs
total_count_top_10 = (
    top_10_comorbidity_ids
    .agg(F.sum('count'))
    .collect()[0][0]
)

# Calculate the percentage for each comorbidity ID in the top 10
top_10_comorbidity_ids_with_percentage = (
    top_10_comorbidity_ids
    .withColumn('percentage', F.col('count') / total_count * 100)
)

# Show the total count of distinct person IDs associated with the top 10 comorbidity IDs
print("Total count of distinct person IDs associated with the top 10 comorbidity IDs:", total_count_top_10)

# Show the top 10 comorbidity IDs along with their counts and percentage
top_10_comorbidity_ids_with_percentage.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+-----+------------------+
|comorbidityid|count|percentage        |
+-------------+-----+------------------+
|S09          |95417|62.60958005249344 |
|S06          |80178|52.61023622047244 |
|Y92          |78699|51.639763779527556|
|Z79          |66350|43.536745406824146|
|M25          |58579|38.43766404199475 |
|I10          |55909|36.68569553805774 |
|M54          |55062|36.12992125984252 |
|Z87          |54963|36.06496062992126 |
|R10          |53180|34.89501312335958 |
|M79          |51965|34.09776902887139 |
+-------------+-----+------------------+



<IPython.core.display.Javascript object>

In [43]:
################################DS - Cohort - Percentage calculated based on the total no of unique personid using Top 10 commo ##
from pyspark.sql import functions as F

# Group by comorbidity ID and count the number of distinct person IDs for each comorbidity ID
comorbidity_counts = (
    Cohort_Demo_Como
    .groupby('comorbidityid')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 comorbidity IDs
top_10_comorbidity_ids = (
    comorbidity_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Calculate the total count of distinct person IDs associated with the top 10 comorbidity IDs
total_count_top_10 = top_10_comorbidity_ids.select(F.sum('count')).collect()[0][0]

# Calculate the percentage for each comorbidity ID in the top 10
top_10_comorbidity_ids_with_percentage = (
    top_10_comorbidity_ids
    .withColumn('percentage', F.col('count') / total_count_top_10 * 100)
)

# Show the total count of distinct person IDs associated with the top 10 comorbidity IDs
print("Total count of distinct person IDs associated with the top 10 comorbidity IDs:", total_count_top_10)

# Show the top 10 comorbidity IDs along with their counts and percentage
top_10_comorbidity_ids_with_percentage.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+-----+------------------+
|comorbidityid|count|percentage        |
+-------------+-----+------------------+
|S09          |95417|14.67272128949319 |
|S06          |80178|12.329348518073141|
|Y92          |78699|12.101915725309166|
|Z79          |66350|10.20295185928999 |
|M25          |58579|9.007968605355664 |
|I10          |55909|8.597390135660048 |
|M54          |55062|8.467142958194808 |
|Z87          |54963|8.451919262127443 |
|R10          |53180|8.17773895820711  |
|M79          |51965|7.990902688289441 |
+-------------+-----+------------------+



<IPython.core.display.Javascript object>

In [42]:
print(Cohort_Demo_Como.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

152400


<IPython.core.display.Javascript object>

In [48]:
##############################Finalized DS : Top 10 CommorbidityId-Control ###############################################
from pyspark.sql import functions as F

# Group by comorbidity ID and count the number of distinct person IDs for each comorbidity ID
comorbidity_counts = (
    Control_Demo_Como
    .groupby('comorbidityid')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 comorbidity IDs
top_10_comorbidity_ids = (
    comorbidity_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Calculate the total count of distinct person IDs present in the entire dataframe
total_count = (
    Control_Demo_Como
    .select(F.countDistinct('personid'))
    .collect()[0][0]
)

# Calculate the total count of distinct person IDs associated with the top 10 comorbidity IDs
total_count_top_10 = (
    top_10_comorbidity_ids
    .agg(F.sum('count'))
    .collect()[0][0]
)

# Calculate the percentage for each comorbidity ID in the top 10
top_10_comorbidity_ids_with_percentage = (
    top_10_comorbidity_ids
    .withColumn('percentage', F.col('count') / total_count * 100)
)

# Show the total count of distinct person IDs associated with the top 10 comorbidity IDs
print("Total count of distinct person IDs associated with the top 10 comorbidity IDs:", total_count_top_10)

# Show the top 10 comorbidity IDs along with their counts and percentage
top_10_comorbidity_ids_with_percentage.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+------+------------------+
|comorbidityid|count |percentage        |
+-------------+------+------------------+
|S09          |707033|70.0821617999669  |
|Y92          |437291|43.34493385127614 |
|S06          |407295|40.3716857492048  |
|Z23          |243589|24.144903718344317|
|Z00          |219662|21.77322391642869 |
|Y93          |213439|21.15639090738782 |
|R51          |212800|21.09305227766307 |
|S01          |201637|19.98655912646217 |
|S00          |199097|19.734790551343444|
|J06          |186007|18.437290296105616|
+-------------+------+------------------+



<IPython.core.display.Javascript object>

In [49]:
###############################DS - Control - Percentage calculated based on the total no of unique personid using Top 10 commo ##
from pyspark.sql import functions as F

# Group by comorbidity ID and count the number of distinct person IDs for each comorbidity ID
comorbidity_counts = (
    Control_Demo_Como
    .groupby('comorbidityid')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 comorbidity IDs
top_10_comorbidity_ids = (
    comorbidity_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Calculate the total count of distinct person IDs associated with the top 10 comorbidity IDs
total_count_top_10 = top_10_comorbidity_ids.select(F.sum('count')).collect()[0][0]

# Calculate the percentage for each comorbidity ID in the top 10
top_10_comorbidity_ids_with_percentage = (
    top_10_comorbidity_ids
    .withColumn('percentage', F.col('count') / total_count_top_10 * 100)
)

# Show the total count of distinct person IDs associated with the top 10 comorbidity IDs
print("Total count of distinct person IDs associated with the top 10 comorbidity IDs:", total_count_top_10)

# Show the top 10 comorbidity IDs along with their counts and percentage
top_10_comorbidity_ids_with_percentage.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+------+------------------+
|comorbidityid|count |percentage        |
+-------------+------+------------------+
|S09          |707033|23.350991627722642|
|Y92          |437291|14.44229403702297 |
|S06          |407295|13.451624089700612|
|Z23          |243589|8.044949386528396 |
|Z00          |219662|7.254718694783427 |
|Y93          |213439|7.049193321994155 |
|R51          |212800|7.028089238238354 |
|S01          |201637|6.65941179384712  |
|S00          |199097|6.575523886586192 |
|J06          |186007|6.143203923576134 |
+-------------+------+------------------+



<IPython.core.display.Javascript object>

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Group by person ID and count the number of distinct comorbidity IDs for each person
person_comorbidity_counts = (
    Cohort_Demo_Como
    .groupby('personid')
    .agg(F.countDistinct('comorbidityid').alias('unique_comorbidity_count'))
)

# Define a window function to rank rows based on the count of distinct comorbidity IDs
window_spec = Window.orderBy(F.desc('unique_comorbidity_count'))

# Select the top row (i.e., person ID with maximum unique comorbidity IDs)
max_unique_comorbidity_person = (
    person_comorbidity_counts
    .withColumn('rank', F.rank().over(window_spec))
    .filter(F.col('rank') == 1)
    .select('personid', 'unique_comorbidity_count')
    .collect()[0]
)

# Display the person ID having maximum unique comorbidity IDs
print("Person ID with maximum unique comorbidity IDs:", max_unique_comorbidity_person['personid'])

# Display the total number of maximum unique comorbidity IDs that person ID has
print("Total number of maximum unique comorbidity IDs:", max_unique_comorbidity_person['unique_comorbidity_count'])

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Person ID with maximum unique comorbidity IDs: aff3f190-9308-48ef-957e-75773a24f94d
Total number of maximum unique comorbidity IDs: 377


<IPython.core.display.Javascript object>

In [18]:
#################Stack only Demo - Med For generating DS #########################################################
from pyspark.sql.functions import when, lit
# Perform left join
Cohort_Demo_Med = Epilepsy_Cohort_Demo.join(Epilepsy_Cohort_Med.select("personid", "drugname"), 
                                       "personid", "left")

# If personid is not present in Epilepsy_Cohort_Commo, mark comorbidityid as null
Cohort_Demo_Med = Cohort_Demo_Med.withColumn("drugname", 
                                 when(Cohort_Demo_Med["drugname"].isNull(), lit(None)).otherwise(Cohort_Demo_Med["drugname"]))

# Show the final result
Cohort_Demo_Med.show(truncate=False)
print("Count of Cohort_Demo_Med", Cohort_Demo_Med.count())

+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+-----+------+-------+-----------------------------------------------+
|personid                            |birthdate |conditioncode|EPI_date                 |TBI_date                 |age_of_TBI_diagnosis|age_at_EPI_diagnosis|race |gender|mh_date|drugname                                       |
+------------------------------------+----------+-------------+-------------------------+-------------------------+--------------------+--------------------+-----+------+-------+-----------------------------------------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|1964-03-10|1            |2015-05-13T12:11:00+00:00|2009-08-13T17:37:44+00:00|45.426705745        |51.17609580333333   |White|Female|69     |Benzodiazepine                                 |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|1964-03-10|1            |2015-05-13T12:11:00+00:00|200

In [19]:
##################################Writing-Stacked-Demo-Med-Cohort############################################################
Cohort_Demo_Med.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Med_Cohort_I1")

In [51]:
################################################DS - Cohort-Demo-Med #################################################
from pyspark.sql import functions as F

# Count the total number of unique drugnames
total_unique_drugnames = Cohort_Demo_Med.select('drugname').distinct().count()

# Group by 'personid' and count the number of drugnames for each person
drugname_counts_per_person = (
    Cohort_Demo_Med
    .groupby('personid')
    .agg(F.countDistinct('drugname').alias('num_drugname'))
)

# Filter out null values before calculating the maximum
non_null_max_drugname = drugname_counts_per_person.filter(F.col('num_drugname').isNotNull()).selectExpr("max(num_drugname) as max_drugname").collect()[0]['max_drugname']

# Calculate the minimum, mean, and standard deviation of the number of drugnames per person
statistics = (
    drugname_counts_per_person
    .select(
        F.min('num_drugname').alias('min_drugname'),
        F.mean('num_drugname').alias('mean_drugname'),
        F.stddev('num_drugname').alias('stddev_drugname')
    )
    .collect()[0]
)

# Display the results
print("Total number of unique drugnames:", total_unique_drugnames)
print("Minimum number of drugnames per person:", statistics['min_drugname'])
print("Maximum number of drugnames per person (excluding null values):", non_null_max_drugname)
print("Mean number of drugnames per person:", statistics['mean_drugname'])
print("Standard deviation of drugnames per person:", statistics['stddev_drugname'])

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total number of unique drugnames: 419
Minimum number of drugnames per person: 0
Maximum number of drugnames per person (excluding null values): 60
Mean number of drugnames per person: 6.459238845144357
Standard deviation of drugnames per person: 6.542764431090104


<IPython.core.display.Javascript object>

In [52]:
#############DS : Cohort-Demo-Med : Top 10 Med - count of pid consuming it and % as total no of pid consuming/total no of pid #########
from pyspark.sql import functions as F

# Filter out null values from the drugname column
Cohort_Demo_Med_filtered = Cohort_Demo_Med.filter(Cohort_Demo_Med.drugname.isNotNull())

# Count the total number of unique person IDs
total_unique_persons = Cohort_Demo_Med_filtered.select('personid').distinct().count()

# Group by drugname and count the number of distinct person IDs for each drugname
drugname_counts = (
    Cohort_Demo_Med_filtered
    .groupby('drugname')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 drug names
top_10_drugnames = (
    drugname_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Calculate the total count of unique person IDs consuming the top 10 drugs
total_count_top_10_drugs = top_10_drugnames.selectExpr("sum(count) as total_count").collect()[0]['total_count']

# Calculate the percentage count for each top 10 drug
top_10_drugnames_with_percentage = (
    top_10_drugnames
    .withColumn('percentage_count', F.col('count') / total_unique_persons * 100)
)

# Show the top 10 drug names along with their count and percentage count
top_10_drugnames_with_percentage.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-----------------------------------+-----+------------------+
|drugname                           |count|percentage_count  |
+-----------------------------------+-----+------------------+
|Opioid Agonist                     |57569|44.62194318490098 |
|Serotonin-3 Receptor Antagonist    |48267|37.41192884548308 |
|Nonsteroidal Anti-inflammatory Drug|42113|32.641940859589965|
|Cephalosporin Antibacterial        |41908|32.483044607216215|
|General Anesthetic                 |30328|23.507344107274346|
|Corticosteroid                     |28427|22.033872030384064|
|Proton Pump Inhibitor              |26801|20.773553462775645|
|beta2-Adrenergic Agonist           |25926|20.095337751424253|
|Antiarrhythmic                     |23812|18.456768592799285|
|beta-Adrenergic Blocker            |22257|17.251482385769094|
+-----------------------------------+-----+------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Group by person ID and count the number of distinct comorbidity IDs for each person
person_comorbidity_counts = (
    Control_Demo_Como
    .groupby('personid')
    .agg(F.countDistinct('comorbidityid').alias('unique_comorbidity_count'))
)

# Define a window function to rank rows based on the count of distinct comorbidity IDs
window_spec = Window.orderBy(F.desc('unique_comorbidity_count'))

# Select the top row (i.e., person ID with maximum unique comorbidity IDs)
max_unique_comorbidity_person = (
    person_comorbidity_counts
    .withColumn('rank', F.rank().over(window_spec))
    .filter(F.col('rank') == 1)
    .select('personid', 'unique_comorbidity_count')
    .collect()[0]
)

# Display the person ID having maximum unique comorbidity IDs
print("Person ID with maximum unique comorbidity IDs:", max_unique_comorbidity_person['personid'])

# Display the total number of maximum unique comorbidity IDs that person ID has
print("Total number of maximum unique comorbidity IDs:", max_unique_comorbidity_person['unique_comorbidity_count'])

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Person ID with maximum unique comorbidity IDs: 0d46602f-6f38-46d3-8e52-4c21a440c8dc
Total number of maximum unique comorbidity IDs: 211


<IPython.core.display.Javascript object>

In [17]:
# Filter records containing personid = 'aff3f190-9308-48ef-957e-75773a24f94d'
filtered_records = Cohort_Demo_Como.filter(Cohort_Demo_Como.personid == 'aff3f190-9308-48ef-957e-75773a24f94d')

# Display the filtered DataFrame
filtered_records.show(truncate=False)
print(filtered_records.count())
print(filtered_records.select("comorbidityid").distinct().count())
print(filtered_records.select("personid").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1


<IPython.core.display.Javascript object>

In [36]:
##############Joining only Demo -Med For DS ##########################################################
from pyspark.sql.functions import when, lit
# Perform left join
Control_Demo_Med = Epilepsy_Control_Demo.join(Epilepsy_Control_Med.select("personid", "drugname"), 
                                       "personid", "left")

# If personid is not present in Epilepsy_Cohort_Commo, mark comorbidityid as null
Control_Demo_Med = Control_Demo_Med.withColumn("drugname", 
                                 when(Control_Demo_Med["drugname"].isNull(), lit(None)).otherwise(Control_Demo_Med["drugname"]))

# Show the final result
Control_Demo_Med.show(truncate=False)
print("Count of Control_Demo_Med", Control_Demo_Med.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of Control_Demo_Med 2925305


<IPython.core.display.Javascript object>

In [ ]:
##################################Writing-Stacked-Demo-Med-Control############################################################
Control_Demo_Med.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Med_Control_I1")

In [53]:
################################################DS - Control-Demo-Med #################################################
from pyspark.sql import functions as F

# Count the total number of unique drugnames
total_unique_drugnames = Control_Demo_Med.select('drugname').distinct().count()

# Group by 'personid' and count the number of drugnames for each person
drugname_counts_per_person = (
    Control_Demo_Med
    .groupby('personid')
    .agg(F.countDistinct('drugname').alias('num_drugname'))
)

# Filter out null values before calculating the maximum
non_null_max_drugname = drugname_counts_per_person.filter(F.col('num_drugname').isNotNull()).selectExpr("max(num_drugname) as max_drugname").collect()[0]['max_drugname']

# Calculate the minimum, mean, and standard deviation of the number of drugnames per person
statistics = (
    drugname_counts_per_person
    .select(
        F.min('num_drugname').alias('min_drugname'),
        F.mean('num_drugname').alias('mean_drugname'),
        F.stddev('num_drugname').alias('stddev_drugname')
    )
    .collect()[0]
)

# Display the results
print("Total number of unique drugnames:", total_unique_drugnames)
print("Minimum number of drugnames per person:", statistics['min_drugname'])
print("Maximum number of drugnames per person (excluding null values):", non_null_max_drugname)
print("Mean number of drugnames per person:", statistics['mean_drugname'])
print("Standard deviation of drugnames per person:", statistics['stddev_drugname'])

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Total number of unique drugnames: 446
Minimum number of drugnames per person: 0
Maximum number of drugnames per person (excluding null values): 58
Mean number of drugnames per person: 2.3810190283517185
Standard deviation of drugnames per person: 4.303662415467396


<IPython.core.display.Javascript object>

In [54]:
#######DS : Control-Demo-Med : Top 10 Med - count of pid consuming it and % as total no of pid consuming/total no of pid #####
from pyspark.sql import functions as F

# Filter out null values from the drugname column
Control_Demo_Med_filtered = Control_Demo_Med.filter(Control_Demo_Med.drugname.isNotNull())

# Count the total number of unique person IDs
total_unique_persons = Control_Demo_Med_filtered.select('personid').distinct().count()

# Group by drugname and count the number of distinct person IDs for each drugname
drugname_counts = (
    Control_Demo_Med_filtered
    .groupby('drugname')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 drug names
top_10_drugnames = (
    drugname_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Calculate the total count of unique person IDs consuming the top 10 drugs
total_count_top_10_drugs = top_10_drugnames.selectExpr("sum(count) as total_count").collect()[0]['total_count']

# Calculate the percentage count for each top 10 drug
top_10_drugnames_with_percentage = (
    top_10_drugnames
    .withColumn('percentage_count', F.col('count') / total_unique_persons * 100)
)

# Show the top 10 drug names along with their count and percentage count
top_10_drugnames_with_percentage.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-----------------------------------+------+------------------+
|drugname                           |count |percentage_count  |
+-----------------------------------+------+------------------+
|Nonsteroidal Anti-inflammatory Drug|166525|34.28697908087629 |
|Serotonin-3 Receptor Antagonist    |145840|30.028001976610113|
|Corticosteroid                     |125220|25.782408169988468|
|Opioid Agonist                     |97588 |20.093065392851262|
|beta2-Adrenergic Agonist           |89888 |18.507659364190413|
|Cephalosporin Antibacterial        |87210 |17.95626750123538 |
|Histamine-1 Receptor Antagonist    |69346 |14.278125514742218|
|Penicillin-class Antibacterial     |69293 |14.267212979739746|
|Antiarrhythmic                     |61883 |12.74151704826223 |
|Proton Pump Inhibitor              |50284 |10.353319057815845|
+-----------------------------------+------+------------------+



<IPython.core.display.Javascript object>

In [38]:
from pyspark.sql import functions as F

# Filter out null values from drugname column
Control_Demo_Med_filtered = Control_Demo_Med.filter(Control_Demo_Med.drugname.isNotNull())

# Group by drugname and count the number of distinct person IDs for each drugname
drugname_counts = (
    Control_Demo_Med_filtered
    .groupby('drugname')
    .agg(F.countDistinct('personid').alias('count'))
)

# Order by count in descending order and select the top 10 drug names
top_10_drugnames = (
    drugname_counts
    .orderBy(F.desc('count'))
    .limit(10)
)

# Show the top 10 drug names
top_10_drugnames.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-----------------------------------+------+
|drugname                           |count |
+-----------------------------------+------+
|Nonsteroidal Anti-inflammatory Drug|166525|
|Serotonin-3 Receptor Antagonist    |145840|
|Corticosteroid                     |125220|
|Opioid Agonist                     |97588 |
|beta2-Adrenergic Agonist           |89888 |
|Cephalosporin Antibacterial        |87210 |
|Histamine-1 Receptor Antagonist    |69346 |
|Penicillin-class Antibacterial     |69293 |
|Antiarrhythmic                     |61883 |
|Proton Pump Inhibitor              |50284 |
+-----------------------------------+------+



<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
from pyspark.sql.functions import col
from tabulate import tabulate

def cohort_distribution_Final_float(spark, cohort_demo):
    cohort_demo = cohort_demo.distinct()
    cohort_demo.cache()

    # Calculate gender counts and percentages
    genderCnt = cohort_demo.select('personid', 'gender').na.drop().distinct().groupBy('gender').count()
    total_gender_count = genderCnt.agg({'count': 'sum'}).collect()[0][0]
    genderCnt = genderCnt.withColumn('percentage', (col('count') / total_gender_count) * 100)

    # Calculate race counts and percentages
    raceCnt = cohort_demo.select('personid', 'race').na.drop().distinct().groupBy('race').count()
    total_race_count = raceCnt.agg({'count': 'sum'}).collect()[0][0]
    raceCnt = raceCnt.withColumn('percentage', (col('count') / total_race_count) * 100)

    # Calculate the total count of distinct individuals before age filtering
    total_individuals_count = cohort_demo.count()
    print('total_individuals_count', total_individuals_count)

    age_ranges = ['age<18', '18<=age<44', '44<=age<60', 'age>=60']  # Age ranges as strings
    age_statistics = []  # To store mean and standard deviation for each age range

    # Calculate counts, percentages, mean, and standard deviation for each age range
    for age_range in age_ranges:
        if age_range == 'age<18':
            age_data = cohort_demo.filter(cohort_demo.age_of_TBI_diagnosis < 18)
        elif age_range == '18<=age<44':
            age_data = cohort_demo.filter((cohort_demo.age_of_TBI_diagnosis >= 18) & (cohort_demo.age_of_TBI_diagnosis < 44))
        elif age_range == '44<=age<60':
            age_data = cohort_demo.filter((cohort_demo.age_of_TBI_diagnosis >= 44) & (cohort_demo.age_of_TBI_diagnosis < 60))
        elif age_range == 'age>=60':
            age_data = cohort_demo.filter(cohort_demo.age_of_TBI_diagnosis >= 60)

        age_count = age_data.count()

        # Calculate percentage using the total count before age filtering
        age_percentage = (age_count / total_individuals_count) * 100

        # Calculate sample statistics
        age_data = age_data.select('age_of_TBI_diagnosis').rdd.flatMap(lambda x: x).collect()
        sample_mean_age = np.mean(age_data)
        sample_std_dev_age = np.std(age_data, ddof=1)

        age_statistics.append([age_range, age_count, f"{age_percentage:.2f}%", f"{sample_mean_age:.2f}", f"{sample_std_dev_age:.2f}"])

    # Calculate mean and standard deviation across all age ranges
    all_age_data = cohort_demo.select('age_of_TBI_diagnosis').rdd.flatMap(lambda x: x).collect()
    all_age_mean = np.mean(all_age_data)
    all_age_std_dev = np.std(all_age_data, ddof=1)

    age_statistics.append(['All Ages', total_individuals_count, '100.00%', f"{all_age_mean:.2f}", f"{all_age_std_dev:.2f}"])

    # Print gender table
    print("Gender Table:")
    print(tabulate(genderCnt.toPandas(), headers=['Gender', 'Count', 'Percentage']))

    # Print race table
    print("Race Table:")
    print(tabulate(raceCnt.toPandas(), headers=['Race', 'Count', 'Percentage']))

    # Print age table
    print("Age Table:")
    print(tabulate(age_statistics, headers=['Age Range', 'Count', 'Percentage', 'Mean Age', 'Standard Deviation'], tablefmt="grid"))

# Example usage
# cohort_demo = spark.read.csv("path/to/your/dataset.csv", header=True, inferSchema=True)
# cohort_distribution_Final_float(cohort_demo)

In [15]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Cohort_toStack")

In [16]:
Epilepsy_Cohort_Lab.printSchema()

root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [17]:
from pyspark.sql.functions import min, max
# Filter records and aggregate to find earliest TBI_date and latest EPI_date
filtered_dates_df1 = Epilepsy_Cohort_Lab.select(
    min("servicedate").alias("earliest_servicedate"),
    max("servicedate").alias("latest_servicedate")
)

# Display the filtered and aggregated DataFrame
filtered_dates_df1.show(truncate=False)

+--------------------+------------------+
|earliest_servicedate|latest_servicedate|
+--------------------+------------------+
|1999-12-18          |2022-08-25        |
+--------------------+------------------+



In [18]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Control_toStack")

In [19]:
Epilepsy_Control_Lab.printSchema()

root
 |-- labcode: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [20]:
from pyspark.sql.functions import min, max
# Filter records and aggregate to find earliest TBI_date and latest EPI_date
filtered_dates_df2 = Epilepsy_Control_Lab.select(
    min("servicedate").alias("earliest_servicedate"),
    max("servicedate").alias("latest_servicedate")
)

# Display the filtered and aggregated DataFrame
filtered_dates_df2.show(truncate=False)

+--------------------+------------------+
|earliest_servicedate|latest_servicedate|
+--------------------+------------------+
|1997-08-23          |2022-08-26        |
+--------------------+------------------+

